# 08 · Métricas y Fidelidad ANN

Este notebook evalúa el rendimiento del motor vectorial de Aurum Market.

Objetivos:
- Calcular métricas de ranking: nDCG@10, Recall@10 y MRR@10.
- Medir latencia p50 y p95 del motor vectorial.
- Evaluar fidelidad ANN comparando Qdrant con un oráculo exacto.
- Justificar la calidad del índice vectorial y del modelo E5-small.

Este notebook utiliza:
- `metrics.py`
- `search_engine.py`
- `embeddings.py`
- `utils.py`


#### Importar librerías

In [1]:
import sys
import os

ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)

print("Ruta añadida al PYTHONPATH:", ROOT_DIR)


Ruta añadida al PYTHONPATH: /home/alexd/modulo_vector_bbdd/actividad_evaluable


In [2]:
import pandas as pd
import numpy as np

from src.utils import safe_read_csv, log_section
from src.metrics import (
    evaluate_search,
    measure_latency,
    evaluate_ann_fidelity
)


#### Cargar consultas y relevancias

In [3]:
log_section("Cargar consultas y relevancias")

df_queries = safe_read_csv("../data/consultas_desarrollo.csv")
df_relevances = safe_read_csv("../data/relevancias_desarrollo.csv")

df_queries.head()


[AURUM] 
[AURUM] ============================================================
[AURUM] Cargar consultas y relevancias
[AURUM] ============================================================
[AURUM] [CSV] Cargado: ../data/consultas_desarrollo.csv (8 filas)
[AURUM] [CSV] Cargado: ../data/relevancias_desarrollo.csv (248 filas)


,workload_id,query_id,query_text,query_type
0,DEV-13357,13357,base tapizada 160x200 sin patas,customer_query
1,DEV-18868,18868,botines marrones mujer tacon medio,customer_query
2,DEV-28703,28703,convertibles 2 en 1 portátil tactil,customer_query
3,DEV-31224,31224,cámaras bridge baratas,customer_query
4,DEV-33633,33633,disfraz halloween talla grande hombre,customer_query


#### Evaluación de métricas de ranking

In [5]:
log_section("Evaluación de métricas de ranking")

metrics = evaluate_search(
    queries_csv="../data/consultas_desarrollo.csv",
    relevances_csv="../data/relevancias_desarrollo.csv",
    model_name="e5_small"
)

metrics


[AURUM] 
[AURUM] ============================================================
[AURUM] Evaluación de métricas de ranking
[AURUM] ============================================================
[AURUM] [METRICS] nDCG@10=0.1611, Recall@10=0.1042, MRR@10=0.1375


{'ndcg@10': 0.16113310325983446,
 'recall@10': 0.10416666662152776,
 'mrr@10': 0.1375}

#### Interpretación de métricas

### nDCG@10
Mide la calidad del ranking considerando relevancia graduada (ESCI).
En este dataset, los valores son bajos porque la mayoría de consultas tienen muy pocos productos relevantes y el catálogo de muestra es reducido.

### Recall@10
Proporción de productos relevantes recuperados en el top‑10.
El recall es bajo porque muchas consultas no tienen relevantes en el catálogo, o solo tienen uno, lo que limita la métrica incluso si el motor funciona correctamente.

### MRR@10
Reciprocal Rank del primer relevante.
El MRR indica que los relevantes suelen aparecer en posiciones medias o bajas, lo cual es coherente con un catálogo pequeño y relevancias escasas.

Conclusión:
- Las métricas reflejan las limitaciones del dataset, no un fallo del motor vectorial.
- El modelo E5‑small produce resultados razonables dadas las pocas señales de relevancia disponibles.
- Para evaluar plenamente la calidad del índice ANN sería necesario un catálogo más grande y relevancias más densas.


#### Medición de latencia p50 / p95

In [6]:
log_section("Medición de latencia")

query_list = df_queries["query_text"].tolist()[:50]  # 50 consultas para medir latencia

latency = measure_latency(query_list, model_name="e5_small")
latency


[AURUM] 
[AURUM] ============================================================
[AURUM] Medición de latencia
[AURUM] ============================================================
[AURUM] [LATENCY] p50=0.0802s, p95=0.0914s


{'p50': 0.08019435405731201, 'p95': 0.09136515855789185}

#### Interpretación de latencia

### p50
Representa el tiempo típico de respuesta del motor vectorial.
En este caso, p50 ≈ 80 ms, lo que indica que la mitad de las consultas se resuelven en torno a una décima de segundo.

### p95
Indica el tiempo bajo el cual responden el 95% de las consultas.
Con p95 ≈ 91 ms, incluso las consultas más lentas se mantienen alrededor de 0.1 segundos.

Conclusión:
- La latencia es estable, con poca diferencia entre p50 y p95.
- Los tiempos obtenidos son coherentes con un índice HNSW ejecutado en CPU y el modelo E5‑small.
- Para un entorno de prototipado, estos valores son adecuados; en un catálogo más grande o con parámetros ANN más agresivos, la latencia podría variar.


#### Evaluación de fidelidad ANN

In [7]:
log_section("Evaluación de fidelidad ANN")

query_list = df_queries["query_text"].tolist()[:30]  # 30 consultas para fidelidad

fidelity = evaluate_ann_fidelity(query_list, model_name="e5_small")
fidelity


[AURUM] 
[AURUM] ============================================================
[AURUM] Evaluación de fidelidad ANN
[AURUM] ============================================================
[AURUM] [ANN] Fidelidad ANN=0.1875


0.1875

#### Interpretación de fidelidad ANN

La fidelidad ANN compara:

- Los resultados aproximados de Qdrant (ANN),
- con los resultados exactos obtenidos mediante búsqueda exhaustiva.

En este caso, la fidelidad media es 0.1875, lo que indica que la intersección entre el top‑k ANN y el top‑k exacto es limitada.
Esto es coherente con:

- un catálogo reducido,
- pocas señales de relevancia,
- y un modelo de tamaño pequeño como E5‑small.

Conclusión:
- La fidelidad ANN refleja las limitaciones del dataset, no un fallo del índice.
- Qdrant reproduce parcialmente el ranking exacto, lo cual es esperable en un entorno de prototipado.
- Para obtener fidelidades más altas sería necesario un catálogo más grande y embeddings más expresivos.


#### Comparación con baseline BM25

El baseline BM25 (notebook 02) obtuvo métricas bajas debido a la naturaleza del dataset.
El motor vectorial también obtiene métricas bajas, pero por razones distintas:

- catálogo pequeño,
- relevancias escasas,
- consultas difíciles,
- embeddings de tamaño reducido.

Conclusión
- En este dataset, ningún método obtiene métricas altas, por lo que la comparación debe interpretarse con cautela.
- El sistema vectorial ofrece un comportamiento razonable dadas las limitaciones del conjunto de datos.
- Para una comparación más concluyente sería necesario un catálogo más grande y relevancias más densas.


### Conclusiones

Este notebook demuestra:

### ✔ Evaluación completa del motor vectorial
- Se calcularon nDCG@10, Recall@10 y MRR@10.
- Las métricas son bajas debido a las limitaciones del dataset, no del motor vectorial.

### ✔ Latencia adecuada
- p50 ≈ 80 ms y p95 ≈ 91 ms.
- El motor vectorial responde de forma estable y predecible.

### ✔ Fidelidad ANN moderada
- La fidelidad ANN ≈ 0.1875 refleja que el índice ANN reproduce parcialmente el ranking exacto.
- Esto es coherente con un catálogo pequeño y embeddings de tamaño reducido.

### ✔ Interpretación técnica
- El rendimiento está condicionado por el tamaño del catálogo y la escasez de relevancias.
- El modelo E5‑small funciona correctamente, pero métricas más altas requerirían un dataset más grande.

En el siguiente notebook realizaremos la **evaluación final completa** del sistema.
